# Career Upskilling Advisor: Member 1 → Member 2

This notebook only **shows the inputs and outputs** of the two components. The logic lives in the GitHub repo.

| Step | Component | Input | Output |
|---|---|---|---|
| 1 | Member 1: skill extraction | job ad (text) | `member1_output.csv` |
| 2 | Member 2: ESCO linking | `member1_output.csv` | `job_skills.json`, `esco_review_required.json`, `member2_candidates.csv` |
| 3 | Member 4: learner model | `job_skills.json` + quiz answers | `learner_estimates.json` |
| 4 | Member 5: course recommender | `job_skills.json` + `learner_estimates.json` + course catalogue | `recommendations.json` (with Coursera links) |

**Kaggle settings:** Internet **On** (clone repo, download models). GPU is optional (faster for Member 1).

In [7]:
!pip install -q transformers sentence-transformers accelerate

## 0. Get the repo and locate the files

In [32]:
import subprocess, sys, shutil, csv, json
from pathlib import Path
import pandas as pd

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

REPO_URL = "https://github.com/Mawa009/Career_Upskilling_Advisor_Plan"
REPO = Path("/kaggle/working/repo")
WORK = Path("/kaggle/working")

if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=False)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)


def find(rel_or_name, search_roots=(REPO, Path("/kaggle/input"))):
    """First match for a file name or relative path, ignoring .git and olderversions."""
    for root in search_roots:
        if not root.exists():
            continue
        for p in root.rglob(Path(rel_or_name).name):
            if ".git" in p.parts or "olderversions" in p.parts:
                continue
            if p.as_posix().endswith(Path(rel_or_name).as_posix()):
                return p
    return None


ESCO_CSV   = find("skills_en.csv")
EXTRACTION = find("src/extraction.py")
LINKING    = find("esco_linking.py")
CACHE_SRC  = find("esco_label_embeddings_cache.npz")   # optional, saves minutes of encoding

missing = [n for n, p in [("skills_en.csv", ESCO_CSV), ("src/extraction.py", EXTRACTION),
                          ("esco_linking.py", LINKING)] if p is None]
assert not missing, f"Not found in the repo: {missing}. Make sure both members' files are pushed."

M1_ROOT = EXTRACTION.parent.parent      # folder that contains src/
M2_DIR  = LINKING.parent
print("ESCO skills :", ESCO_CSV)
print("Member 1    :", M1_ROOT)
print("Member 2    :", LINKING)
print("Embed cache :", CACHE_SRC or "not found (will be built, a few minutes)")

ESCO skills : /kaggle/working/repo/memberB_AIcon26/memberB_AIcon26/skills_en.csv
Member 1    : /kaggle/working/repo/member1
Member 2    : /kaggle/working/repo/memberB_AIcon26/memberB_AIcon26/esco_linking.py
Embed cache : /kaggle/working/repo/memberB_AIcon26/memberB_AIcon26/esco_label_embeddings_cache.npz


## 1. Member 1: skill extraction
### Input: job ad

In [9]:
JOB_AD = (
    "We need a Data Analyst proficient in Python, SQL and Power BI. "
    "Build dashboards, analyze customer data and perform statistical analysis. "
    "Experience with TensorFlow, LangChain and Next.js is a plus."
)
print(JOB_AD)

We need a Data Analyst proficient in Python, SQL and Power BI. Build dashboards, analyze customer data and perform statistical analysis. Experience with TensorFlow, LangChain and Next.js is a plus.


### Run Member 1

In [10]:
sys.path.insert(0, str(M1_ROOT))
from src.extraction import SkillExtractor

extractor = SkillExtractor(str(ESCO_CSV))
skills = extractor.extract_detailed(JOB_AD)
print(f"Extracted {len(skills)} skills")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Extracted 9 skills


### Output: `member1_output.csv` (handed to Member 2)

In [11]:
M1_CSV = WORK / "member1_output.csv"
COLS = ["skill", "count", "sources", "confidence", "requirement", "importance_hint", "esco_candidates"]

with M1_CSV.open("w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=COLS)
    w.writeheader()
    for s in skills:
        d = s.to_dict()
        w.writerow({
            "skill": d["skill"],
            "count": d["count"],
            "sources": "|".join(d["sources"]),
            "confidence": d["confidence"],
            "requirement": d["requirement"],
            "importance_hint": d["importance_hint"],
            "esco_candidates": "|".join(d["esco_candidates"]),
        })

pd.read_csv(M1_CSV)

,skill,count,sources,confidence,requirement,importance_hint,esco_candidates
0,Python,1,jobbert_knowledge|tech,1.0000,required,1.0,NaN
1,SQL,1,jobbert_knowledge|tech,1.0000,required,1.0,NaN
2,Power BI,1,esco|jobbert_knowledge|tech,1.0000,required,1.0,data visualisation software
3,Build dashboards,1,jobbert_skill,1.0000,unknown,0.7,NaN
4,analyze customer data,1,jobbert_skill,1.0000,unknown,0.7,NaN
5,perform statistical analysis,1,esco|jobbert_skill,0.9923,unknown,0.7,statistics
6,TensorFlow,1,jobbert_knowledge|tech,1.0000,preferred,0.5,NaN
7,LangChain,1,jobbert_knowledge|tech,1.0000,preferred,0.5,NaN
8,Next.js,1,jobbert_knowledge|tech,1.0000,preferred,0.5,NaN


## 2. Member 2: ESCO linking
### Input: `member1_output.csv`

In [12]:
pd.read_csv(M1_CSV)[["skill", "requirement", "importance_hint", "confidence"]]

,skill,requirement,importance_hint,confidence
0,Python,required,1.0,1.0000
1,SQL,required,1.0,1.0000
2,Power BI,required,1.0,1.0000
3,Build dashboards,unknown,0.7,1.0000
4,analyze customer data,unknown,0.7,1.0000
5,perform statistical analysis,unknown,0.7,0.9923
6,TensorFlow,preferred,0.5,1.0000
7,LangChain,preferred,0.5,1.0000
8,Next.js,preferred,0.5,1.0000


### Run Member 2
`esco_linking.py` is used as-is. Only its Windows paths and its JSON input loader are redirected so it reads Member 1's CSV on Kaggle.

In [13]:
sys.path.insert(0, str(M2_DIR))
import esco_linking as m2

# Paths (original file points to D:\\AIcon26)
m2.ESCO_FILE   = ESCO_CSV
m2.INPUT_FILE  = M1_CSV
m2.CACHE_FILE  = WORK / "esco_label_embeddings_cache.npz"
m2.OUTPUT_CSV  = WORK / "member2_candidates.csv"
m2.OUTPUT_JSON = WORK / "job_skills.json"
m2.REVIEW_JSON = WORK / "esco_review_required.json"
if CACHE_SRC and not m2.CACHE_FILE.exists():
    shutil.copy(CACHE_SRC, m2.CACHE_FILE)


# Input adapter: Member 1 CSV -> the dicts Member 2's pipeline expects
def load_skills_from_csv():
    results, seen = [], set()
    with m2.INPUT_FILE.open("r", encoding="utf-8-sig", newline="") as f:
        for row in csv.DictReader(f):
            skill = m2.clean(row.get("skill"))
            key = m2.normalize(skill)
            if not key or key in seen:
                continue
            seen.add(key)
            importance, source = m2.get_importance(row)   # uses importance_hint / requirement
            results.append({
                "text": skill,
                "importance": importance,
                "importance_source": source,
                "requirement": m2.clean(row.get("requirement")),
                "skill_confidence": row.get("confidence", ""),
            })
    return results


m2.load_skills = load_skills_from_csv
m2.main()

Loading ESCO...
Loaded 13939 ESCO concepts.
Loaded 9 input skills.
Loading model: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Using cached ESCO embeddings.
Python: EXACT_ALIAS
SQL: EXACT_LABEL
Power BI: EXACT_ALIAS
Build dashboards: REVIEW_REQUIRED
analyze customer data: REVIEW_REQUIRED
perform statistical analysis: REVIEW_REQUIRED
TensorFlow: REVIEW_REQUIRED
LangChain: REVIEW_REQUIRED
Next.js: REVIEW_REQUIRED

Finished.
Input skills: 9
Accepted exact mappings: 3
Items needing review: 6
Candidate CSV: /kaggle/working/member2_candidates.csv
Team handoff JSON: /kaggle/working/job_skills.json
Review file: /kaggle/working/esco_review_required.json

Note: semantic matches are deliberately excluded from job_skills.json until a human confirms them.


### Output 1: accepted mappings (`job_skills.json`, team contract)

In [14]:
accepted = json.loads((WORK / "job_skills.json").read_text(encoding="utf-8"))
pd.DataFrame(accepted, columns=["esco_id", "label", "importance"])

,esco_id,label,importance
0,http://data.europa.eu/esco/skill/ccd0a1d9-afda-43d9-b901-96344886e14d,Python (computer programming),1.0
1,http://data.europa.eu/esco/skill/598de5b0-5b58-4ea7-8058-a4bc4d18c742,SQL,1.0
2,http://data.europa.eu/esco/skill/65e58886-bd1e-4c5b-8ca5-8d9b353c8aa1,data visualisation software,1.0


### Output 2: needs human review (`esco_review_required.json`)

In [15]:
review = json.loads((WORK / "esco_review_required.json").read_text(encoding="utf-8"))
rows = []
for r in review:
    cands = r.get("candidates") or []
    rows.append({
        "input_skill": r["input_skill"],
        "status": r.get("status") or r.get("match_status"),
        "suggested": r.get("label") or (cands[0]["label"] if cands else None),
        "top_candidates": " | ".join(f'{c["label"]} ({c["similarity_score"]})' for c in cands),
    })
pd.DataFrame(rows)

,input_skill,status,suggested,top_candidates
0,Build dashboards,REVIEW_REQUIRED,install construction profiles,install construction profiles (0.4806) | build recommender systems (0.4617) ...
1,analyze customer data,REVIEW_REQUIRED,collect customer data,collect customer data (0.8182) | analyse data about clients (0.7586) | asses...
2,perform statistical analysis,REVIEW_REQUIRED,apply statistical analysis techniques,apply statistical analysis techniques (0.8208) | perform data analysis (0.79...
3,TensorFlow,REVIEW_REQUIRED,computer vision,computer vision (None) | deep learning (0.5366) | artificial neural networks...
4,LangChain,REVIEW_REQUIRED,Javanese,Javanese (0.4894) | Dutch (0.4775) | Turkish (0.4746)
5,Next.js,REVIEW_REQUIRED,JavaScript,JavaScript (0.5061) | JSSS (0.491) | JavaScript Framework (0.4026)


### Output 3: candidate audit (`member2_candidates.csv`, selected rows)

In [16]:
cand = pd.read_csv(WORK / "member2_candidates.csv", encoding="utf-8-sig")
cand[cand["candidate_selected"]][
    ["input_skill", "esco_label", "match_method", "match_status", "importance_score"]
]

,input_skill,esco_label,match_method,match_status,importance_score
0,Python,Python (computer programming),EXACT_ALIAS,EXACT_ALIAS,1.0
3,SQL,SQL,EXACT_LABEL,EXACT_LABEL,1.0
6,Power BI,data visualisation software,EXACT_ALIAS,EXACT_ALIAS,1.0


## 3. Member 4: how well does the learner know each skill?
**Input:** Member 2's `job_skills.json` + the learner's quiz answers. **Output:** `learner_estimates.json` = `{esco_id: knowledge 0..1}`.

Knowledge is hidden, so it is estimated with Bayesian Knowledge Tracing (BKT): after every answer the estimate is updated, and the model's *prediction of the next answer* is how we check it is sensible.

In [17]:
# Load Member 4 by file path (Member 1 and Member 4 both have a package named `src`)
import importlib.util, random
from datetime import datetime, timedelta, timezone

LEARNER_MODEL = find("src/learner_model.py")
assert LEARNER_MODEL, "Member 4 files not found in the repo: push the member4_llm_live folder."
M4 = LEARNER_MODEL.parent.parent


def load_m4(name):
    spec = importlib.util.spec_from_file_location(f"m4_{name}", M4 / "src" / f"{name}.py")
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


lm, qe, ja = load_m4("learner_model"), load_m4("quiz_engine"), load_m4("job_assessment")
print("Member 4:", M4)

Member 4: /kaggle/working/repo/member4_llm_live


### Input: Member 2's skills, and which ones Member 4 can quiz

In [18]:
job_skills = ja.parse_job_skills(json.loads((WORK / "job_skills.json").read_text(encoding="utf-8")))
questions  = qe.load_questions(M4 / "data" / "questions.json")
esco_label = {c["uri"]: c["label"] for c in m2.load_esco()}
plan = ja.make_coverage_plan(job_skills, questions, esco_label)

skill_ids = ja.assessable_ids(plan)
assert skill_ids, "None of Member 2's skills have questions in Member 4's bank."
label_of = {s["esco_id"]: s["label"] for s in job_skills}
pd.DataFrame(plan)[["label", "importance", "question_count", "coverage"]]

,label,importance,question_count,coverage
0,Python (computer programming),1.0,12,assessable
1,SQL,1.0,12,assessable
2,data visualisation software,1.0,12,provisional_mapping


### Quiz answers
Answers below are **simulated** (the learner knows each skill with probability `KNOWS`) so the notebook runs without a person. For a real learner, replace the `chosen = ...` line with the option index they picked.

In [19]:
PER_SKILL, KNOWS, rng = 5, 0.75, random.Random(0)
quiz = qe.select_questions(questions, skill_ids, per_skill=PER_SKILL, seed=0)
start = datetime(2026, 10, 10, 12, 0, tzinfo=timezone.utc)

history = []
for i, q in enumerate(quiz):
    wrong = [k for k in range(len(q["options"])) if k != q["correct_answer"]]
    chosen = q["correct_answer"] if rng.random() < KNOWS else rng.choice(wrong)   # <- real learner's choice
    history.append(qe.make_interaction("L001", q, chosen, (start + timedelta(minutes=i)).isoformat()))

hist_df = pd.DataFrame(history)
hist_df.insert(1, "skill", hist_df["esco_id"].map(label_of))
hist_df[["question_id", "skill", "correct", "timestamp"]]

,question_id,skill,correct,timestamp
0,SQL_007,SQL,0,2026-10-10T12:00:00+00:00
1,SQL_004,SQL,1,2026-10-10T12:01:00+00:00
2,PYT_008,Python (computer programming),0,2026-10-10T12:02:00+00:00
3,SQL_003,SQL,1,2026-10-10T12:03:00+00:00
4,SQL_002,SQL,0,2026-10-10T12:04:00+00:00
5,PYT_004,Python (computer programming),0,2026-10-10T12:05:00+00:00
6,PYT_007,Python (computer programming),1,2026-10-10T12:06:00+00:00
7,PYT_001,Python (computer programming),0,2026-10-10T12:07:00+00:00
8,PYT_002,Python (computer programming),1,2026-10-10T12:08:00+00:00
9,SQL_008,SQL,1,2026-10-10T12:09:00+00:00


### Output: knowledge estimate per skill (`learner_estimates.json`)

In [20]:
learner_estimates = lm.estimate_skill_knowledge("L001", history)                    # BKT
accuracy          = lm.estimate_skill_knowledge("L001", history, method="accuracy") # simple baseline
(WORK / "learner_estimates.json").write_text(json.dumps(learner_estimates, indent=2), encoding="utf-8")

rows = []
for s in job_skills:
    u = s["esco_id"]
    ev = [h["correct"] for h in history if h["esco_id"] == u]
    rows.append({"skill": s["label"], "importance": s["importance"], "answered": len(ev), "correct": sum(ev),
                 "knowledge (BKT)": learner_estimates.get(u, "not assessed"),
                 "smoothed accuracy": accuracy.get(u, "not assessed")})
pd.DataFrame(rows)

,skill,importance,answered,correct,knowledge (BKT),smoothed accuracy
0,Python (computer programming),1.0,5,2,0.0167,0.4286
1,SQL,1.0,5,3,0.379,0.5714
2,data visualisation software,1.0,0,0,not assessed,not assessed


### Check: predicting the next answer
Before each answer the model predicts `P(correct)` from what it knew so far, then updates. Few answers and simulated data only prove the pipeline works; a real evaluation needs held-out answers from real learners.

In [21]:
p, state, preds = lm.DEFAULT_BKT, {}, []
for h in sorted(history, key=lambda h: h["timestamp"]):
    u = h["esco_id"]
    k = state.get(u, p["prior"])
    pred = lm.predict_correct_probability(k, p["guess"], p["slip"])
    preds.append({"skill": label_of[u], "question": h["question_id"],
                  "P(correct) predicted": round(pred, 3), "actual": h["correct"]})
    state[u] = lm.bkt_update(k, h["correct"], p["guess"], p["slip"], p["learn"])

pred_df = pd.DataFrame(preds)
hit   = ((pred_df["P(correct) predicted"] >= 0.5) == (pred_df["actual"] == 1)).mean()
brier = ((pred_df["P(correct) predicted"] - pred_df["actual"]) ** 2).mean()
print(f"Next-answer accuracy: {hit:.0%}   Brier score: {brier:.3f} (lower is better)")
pred_df

Next-answer accuracy: 40%   Brier score: 0.356 (lower is better)


,skill,question,P(correct) predicted,actual
0,SQL,SQL_007,0.410,0
1,SQL,SQL_004,0.236,1
2,Python (computer programming),PYT_008,0.410,0
3,SQL,SQL_003,0.336,1
4,SQL,SQL_002,0.564,0
5,Python (computer programming),PYT_004,0.236,0
6,Python (computer programming),PYT_007,0.205,1
7,Python (computer programming),PYT_001,0.220,0
8,Python (computer programming),PYT_002,0.203,1
9,SQL,SQL_008,0.284,1


## 4. Member 5: which courses close the gaps?
**Inputs:** Member 2's `job_skills.json`, Member 4's `learner_estimates.json`, and a course catalogue. **Output:** `recommendations.json` = `[{course_id, score, reason, title, hours, url}]` (the first three are Member 5's contract; title, hours and the Coursera link are added).

Gap per skill: `priority = importance × (1 − mastery)`. Courses are ranked by gap closed per hour, respecting prerequisites and difficulty, and skills already covered by a chosen course are not counted twice.

In [22]:
RECS = find("src/recommendations.py")
assert RECS, "Member 5 files not found in the repo: push src/ and data/mock/ from member5_deliverables."
sys.path.insert(0, str(RECS.parent))          # Member 5 uses flat imports (`from skill_gaps import ...`)
from recommendations import recommend_courses
from skill_gaps import compute_skill_gaps

### Inputs: skill gaps (from Members 2 and 4)

In [23]:
job_skills        = json.loads((WORK / "job_skills.json").read_text(encoding="utf-8"))          # Member 2
learner_estimates = json.loads((WORK / "learner_estimates.json").read_text(encoding="utf-8"))   # Member 4

gaps = pd.DataFrame(compute_skill_gaps(job_skills, learner_estimates))
gaps["assessed"] = gaps["esco_id"].isin(learner_estimates)   # not assessed => Member 5 treats mastery as 0
gaps[["label", "importance", "mastery", "priority", "assessed"]]

,label,importance,mastery,priority,assessed
0,data visualisation software,1.0,0.0000,1.000,False
1,Python (computer programming),1.0,0.0167,0.983,True
2,SQL,1.0,0.3790,0.621,True


### Course catalogue, tagged directly to the job's ESCO skills
Member 5's catalogue was tagged against only 5 hand-picked skills, so any other skill would have no courses. Here the Coursera dataset is re-tagged for **whichever ESCO skills the job needs**:
a course skill counts as an ESCO skill if it equals its label or an ESCO alias (exact), or is semantically close to its label (cosine ≥ `THRESHOLD`, untuned, raise it for stricter tags).
If the dataset or model cannot be downloaded, it falls back to Member 5's 5-skill catalogue.

In [24]:
import re, ast
import numpy as np

THRESHOLD = 0.70     # cosine similarity for "this course skill means this ESCO skill"


def schedule_to_hours(text):                       # from Member 5's notebook
    if not isinstance(text, str):
        return None
    t = text.lower()
    m = re.search(r"(\d+(?:\.\d+)?)\s*hours?\s*to complete", t)
    if m:
        return float(m.group(1))
    m = re.search(r"(\d+(?:\.\d+)?)\s*(month|week)s?\s*\(at\s*(\d+(?:\.\d+)?)\s*hours?", t)
    if m:
        n, unit, per_week = float(m.group(1)), m.group(2), float(m.group(3))
        return (n * 4 if unit == "month" else n) * per_week
    return None


def clean_skills(text):                            # from Member 5's notebook
    try:
        items = ast.literal_eval(text) if isinstance(text, str) else []
    except (ValueError, SyntaxError):
        return []
    seen, out = set(), []
    for x in items:
        x = str(x).strip().lower().replace("|", " ")
        if x and x not in seen:
            seen.add(x)
            out.append(x)
    return out


def build_catalogue(job_skills, threshold=THRESHOLD):
    from datasets import load_dataset
    from sentence_transformers import SentenceTransformer

    df = load_dataset("azrai99/coursera-course-dataset")["train"].to_pandas()
    df["estimated_hours"] = df["Schedule"].apply(schedule_to_hours)
    df["skill_list"] = df["Skills"].apply(clean_skills)
    df = df[df["estimated_hours"].notna() & (df["skill_list"].str.len() > 0)].copy()

    concepts = {c["uri"]: c for c in m2.load_esco()}
    exact = {}                                     # phrase -> ESCO uris (label + aliases)
    for s in job_skills:
        names = [s["label"]] + concepts.get(s["esco_id"], {}).get("aliases", [])
        for name in names:
            exact.setdefault(name.lower(), set()).add(s["esco_id"])

    uniq = sorted({x for lst in df["skill_list"] for x in lst})
    model = SentenceTransformer(m2.MODEL_NAME)
    enc = lambda xs: model.encode(xs, batch_size=256, convert_to_numpy=True,
                                  normalize_embeddings=True, show_progress_bar=False)
    sim = enc([s["label"].lower() for s in job_skills]) @ enc(uniq).T      # job skills x course skills

    tags = {}                                      # course skill text -> ESCO uris
    for j, text in enumerate(uniq):
        hit = set(exact.get(text, set()))
        hit |= {job_skills[i]["esco_id"] for i in np.where(sim[:, j] >= threshold)[0]}
        if hit:
            tags[text] = hit

    df["esco_skill_ids"] = df["skill_list"].apply(
        lambda lst: "|".join(sorted(set().union(*[tags.get(x, set()) for x in lst]))))
    df = df[df["esco_skill_ids"] != ""].reset_index(drop=True)
    return pd.DataFrame({
        "course_id": [f"r{i:04d}" for i in range(len(df))],
        "title": df["title"], "description": df["Description"].fillna("").str.slice(0, 200),
        "esco_skill_ids": df["esco_skill_ids"],
        "difficulty": df["Level"].fillna("").str.replace(" level", "", regex=False).str.lower(),
        "estimated_hours": df["estimated_hours"], "prerequisites": "",
        "resource_url": df["URL"].fillna(""),
    })


def fallback_catalogue():                          # Member 5's catalogue: only 5 skills
    ESCO = "http://data.europa.eu/esco/skill/"
    mapping = {
        "skill_sql":    ESCO + "598de5b0-5b58-4ea7-8058-a4bc4d18c742",   # SQL
        "skill_python": ESCO + "ccd0a1d9-afda-43d9-b901-96344886e14d",   # Python (computer programming)
        "skill_viz":    ESCO + "65e58886-bd1e-4c5b-8ca5-8d9b353c8aa1",   # data visualisation software
        "skill_stats":  ESCO + "7ee4c2ea-b349-4bd2-81a3-ec31475d4833",   # statistics
        "skill_excel":  ESCO + "1973c966-f236-40c9-b2d4-5d71a89019be",   # use spreadsheets software
    }
    path = find("courses_real_v2.csv")
    assert path, "courses_real_v2.csv not found in the repo."
    cat = pd.read_csv(path, keep_default_na=False)
    cat["esco_skill_ids"] = cat["esco_skill_ids"].map(lambda ids: "|".join(mapping[x] for x in ids.split("|")))
    return cat


try:
    catalogue = build_catalogue(job_skills)
    print(f"Tagged {len(catalogue)} courses directly to the job's {len(job_skills)} ESCO skills")
except Exception as exc:
    print(f"Could not build the catalogue ({type(exc).__name__}: {exc}).")
    print("Using Member 5's 5-skill catalogue instead. Skills outside those 5 will have no courses.")
    catalogue = fallback_catalogue()

label_of_uri = {s["esco_id"]: s["label"] for s in job_skills}
counts = {u: 0 for u in label_of_uri}
for ids in catalogue["esco_skill_ids"]:
    for u in ids.split("|"):
        if u in counts:
            counts[u] += 1
pd.DataFrame({"skill": [label_of_uri[u] for u in counts], "courses": list(counts.values())})

README.md:   0%|          | 0.00/355 [00:00<?, ?B/s]

coursera_course_2024.csv: reconstructing file:   0%|          |  0.00B / 23.4MB            

coursera_course_2024.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/6645 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Tagged 331 courses directly to the job's 3 ESCO skills


,skill,courses
0,Python (computer programming),191
1,SQL,51
2,data visualisation software,118


### Output: recommended courses (`recommendations.json`)

In [25]:
recs = recommend_courses(job_skills, learner_estimates, catalogue, completed=set(), top_n=5)

# Member 5's contract is {course_id, score, reason}; add title, hours and the Coursera link (extra fields only)
info = catalogue.set_index("course_id")
for r in recs:
    r["title"] = info.loc[r["course_id"], "title"]
    r["hours"] = float(info.loc[r["course_id"], "estimated_hours"])
    r["url"]   = info.loc[r["course_id"], "resource_url"]
(WORK / "recommendations.json").write_text(json.dumps(recs, indent=2, ensure_ascii=False), encoding="utf-8")

rec_df = pd.DataFrame(recs, columns=["course_id", "title", "hours", "score", "reason", "url"])
rec_df

,course_id,title,hours,score,reason,url
0,r0310,Power BI Data Analyst Associate Prep,3.0,0.3242,"Closes new gap: SQL (priority 0.621), data visualisation software (priority ...",https://www.coursera.org/learn/power-bi-data-analyst-associate-prep
1,r0300,Python for Active Defense,4.0,0.1966,Closes new gap: Python (computer programming) (priority 0.983) in 4 hours,https://www.coursera.org/learn/python-for-active-defense


### Files produced in `/kaggle/working`
`member1_output.csv` · `job_skills.json` · `esco_review_required.json` · `member2_candidates.csv` · `learner_estimates.json` · `recommendations.json`

In [26]:
for n in ["member1_output.csv", "job_skills.json", "esco_review_required.json", "member2_candidates.csv", "learner_estimates.json", "recommendations.json"]:
    p = WORK / n
    print(f"{n:30s} {p.stat().st_size:>8,d} bytes" if p.exists() else f"{n:30s} MISSING")

member1_output.csv                  615 bytes
job_skills.json                     469 bytes
esco_review_required.json         3,924 bytes
member2_candidates.csv            9,776 bytes
learner_estimates.json              167 bytes
recommendations.json                596 bytes


In [34]:
# ===== Member 3: which jobs do people usually move to next? (JobHop career transitions) =====
import importlib, importlib.util, sys
import pandas as pd

# Member 3's code is a package called `src` (same name as Members 1 and 4), so load it under another name.
CAREER_PATHS = find("career_paths.py")
assert CAREER_PATHS, "Member 3's code not found in the repo: push the member3_career_transitions folder."
M3_SRC = CAREER_PATHS.parent
spec = importlib.util.spec_from_file_location("m3_src", M3_SRC / "__init__.py",
                                              submodule_search_locations=[str(M3_SRC)])
pkg = importlib.util.module_from_spec(spec)
sys.modules["m3_src"] = pkg
spec.loader.exec_module(pkg)
m3 = importlib.import_module("m3_src.inference")

# Member 3's trained model files (both must sit in the same folder)
SECOND_ORDER = find("second_order_counts.json")
assert SECOND_ORDER and (SECOND_ORDER.parent / "transition_counts.json").exists(), \
    "Push Member 3's transition_counts.json and second_order_counts.json (same folder)."
m3.MODEL_DIR = SECOND_ORDER.parent

# Optional, for readable job names: ESCO's occupations_en.csv (same download as skills_en.csv)
OCC_CSV = find("occupations_en.csv")
occ, name = None, {}
if OCC_CSV:
    occ  = pd.read_csv(OCC_CSV, dtype=str).dropna(subset=["code"]).drop_duplicates("code")
    name = dict(zip(occ["code"], occ["preferredLabel"]))
else:
    print("occupations_en.csv not found: showing ESCO occupation codes only.")


def find_job_code(title, limit=5):
    """ESCO occupation codes whose name contains `title` (needs occupations_en.csv)."""
    assert occ is not None, "Needs occupations_en.csv in the repo."
    return occ.loc[occ["preferredLabel"].str.contains(title, case=False, regex=False),
                   ["code", "preferredLabel"]].head(limit)


def next_jobs(current, previous=None, top_n=5):
    """Jobs people most often move to after `current` (ESCO occupation code).
    Pass `previous` (the job before it) to use Member 3's history-aware second-order model."""
    recs = m3.predict_next_occupations(current, previous, top_k=top_n)
    if not recs:
        print(f"No reliable data for {current} (too few observed moves).")
        return pd.DataFrame()
    out = pd.DataFrame({"next_job_code": [r["occupation_id"] for r in recs],
                        "next_job": [name.get(r["occupation_id"], "") for r in recs],
                        "probability": [round(r["probability"], 3) for r in recs]})
    return out if name else out.drop(columns="next_job")

In [53]:
# Take job title as input
job_title = input("Enter a job title: ")

# Find matching ESCO occupation
job_code = find_job_code(job_title)

# Check whether any matching job was found
if not job_code.empty:

    print(job_code)

    # If the function returns a DataFrame, inspect its columns
    print("Available columns:", job_code.columns.tolist())

else:
    print("Job title not found.")

Enter a job title:  data


        code           preferredLabel
269   2521.5  data warehouse designer
277   2619.4  data protection officer
413  2511.20            data engineer
481   2511.4           data scientist
771   2521.4      database integrator
Available columns: ['code', 'preferredLabel']
